In [1]:
#import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder

import pickle

from sklearn.metrics import mean_squared_error, r2_score

In [2]:
#load the customer churn dataset
data = pd.read_csv('Churn_Modelling.csv')

In [3]:
#EDA => understanding dataset before training the model
data.info()

print("Missing values:")
print(data.isnull().sum())

print("Salary summary:")
print(data["EstimatedSalary"].describe())

print("Correlations with salary:")
print(
    data.drop(
        columns=["RowNumber", "CustomerId", "Surname"]
    )
    .corr(numeric_only=True)["EstimatedSalary"]
    .sort_values(ascending=False)
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB
Missing values:
RowNumber          0
CustomerId         0
Surname            0
CreditScore    

In [4]:
## preprocessing the dataset
data = data.drop(['RowNumber','CustomerId','Surname'],axis = 1)

X = data.drop(
    "EstimatedSalary",axis = 1
)

y=data['EstimatedSalary']

In [5]:
#Train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

##Here the train test split only works on the training data while test data remains unchanged
X_train,X_val,y_train,y_val = train_test_split(X_train,y_train,test_size = 0.2,random_state = 42)
data.head()


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
#Feature Engineering : converting textual data like gender and geography to numbers

label_encoder_gender = LabelEncoder()
X_train['Gender'] = label_encoder_gender.fit_transform(X_train['Gender'])
X_test['Gender'] = label_encoder_gender.transform(X_test['Gender'])


X_val['Gender'] = label_encoder_gender.transform(X_val['Gender'])



OneHotEncoder_geography = OneHotEncoder()
geo_encoder = OneHotEncoder_geography.fit_transform(X_train[['Geography']]).toarray()
geo_encoder_test = OneHotEncoder_geography.transform(X_test[['Geography']]).toarray()


geo_encoder_val = OneHotEncoder_geography.transform(X_val[['Geography']]).toarray()


print(geo_encoder)



geo_encoder_df= pd.DataFrame(
    geo_encoder,
    columns = OneHotEncoder_geography.get_feature_names_out(['Geography']),
    index = X_train.index

)

geo_encoder_test_df= pd.DataFrame(
    geo_encoder_test,
    columns = OneHotEncoder_geography.get_feature_names_out(['Geography']),
    index = X_test.index

)

print(geo_encoder_df.head())



geo_encoder_val_df= pd.DataFrame(
    geo_encoder_val,
    columns = OneHotEncoder_geography.get_feature_names_out(['Geography']),
    index = X_val.index

)








X_train = pd.concat(
    [
        X_train.drop(['Geography'], axis = 1),
        geo_encoder_df
    ],
    axis = 1
)


X_test = pd.concat(
    [
        X_test.drop(['Geography'], axis = 1),
        geo_encoder_test_df
    ],
    axis = 1
)




X_val = pd.concat(
    [
        X_val.drop(['Geography'], axis = 1),
        geo_encoder_val_df
    ],
    axis = 1
)



with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender,file)


with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(OneHotEncoder_geography,file)

print("Encoders saved successfuly")

[[0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 ...
 [1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]
      Geography_France  Geography_Germany  Geography_Spain
2579               0.0                1.0              0.0
7128               0.0                0.0              1.0
4082               0.0                1.0              0.0
2550               1.0                0.0              0.0
4926               0.0                1.0              0.0
Encoders saved successfuly


In [7]:
## Using standard scaler to bring the values between 0 and 1
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [8]:
## writing the results to a pickle file so that it can be later loaded
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

print("Scaler saved successfully")

Scaler saved successfully


In [9]:
## Build ANN model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

import datetime
input_shape = (X_train.shape[1],)
print("ANN input shape: ",input_shape)


model = Sequential([
    Dense(64,activation = 'relu',input_shape=input_shape),
    Dense(32,activation = 'relu'),
    Dense(1,activation = 'linear') # output  layer
])



model.summary()

I0000 00:00:1792475448.709328   64017 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792475448.794541   64017 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792475451.908780   64017 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ANN input shape:  (12,)


/home/sheheryar/Desktop/Salary_Prediction App/churn1/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1792475453.337245   64017 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
## Model Training ANN

optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01)
# Adam is used to help the neural network learn by reducing its prediction errors. It updates the model’s weights and biases during training.


loss_function = tf.keras.losses.MeanAbsoluteError()

print(loss_function)


model.compile(
    optimizer = optimizer,
    loss = loss_function,
    metrics = ["mae"]
)

log_dir = "logs/fit/"+ datetime.datetime.now().strftime("%Y%m%d")


tensorboard_callback = TensorBoard(
    log_dir = log_dir,
    histogram_freq = 1
)

print("Tensorboard log directory: ",log_dir)

early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),    epochs = 100,
    callbacks = [tensorboard_callback,early_stopping_callback]
)

<LossFunctionWrapper(<function mean_absolute_error at 0x79ed1e5c51b0>, kwargs={})>
Tensorboard log directory:  logs/fit/20261020
Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 85792.7109 - mae: 85792.7109 - val_loss: 55262.2148 - val_mae: 55262.2148
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 50691.3008 - mae: 50691.3008 - val_loss: 50502.1719 - val_mae: 50502.1719
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 49993.3516 - mae: 49993.3516 - val_loss: 50319.3984 - val_mae: 50319.3984
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 49816.6836 - mae: 49816.6836 - val_loss: 50108.4648 - val_mae: 50108.4648
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 49695.1602 - mae: 49695.1602 - val_loss: 49927.9766 - val_mae: 49927.9766
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 49623.4883 - mae: 49623.4883 - val_loss: 49878.4141 - val_mae: 49878.4141
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 

In [11]:
## Evaluating the model
test_loss,test_mae = model.evaluate(X_test,y_test,verbose = 0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

Test Loss: 50482.9883
Test MAE: 50482.9883


In [12]:
## Calculate RMSE and R2
y_pred = model.predict(X_test, verbose=0).flatten()

test_rmse = mean_squared_error(y_test, y_pred) ** 0.5
test_r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test R2: {test_r2:.4f}")

Test RMSE: 58428.8612
Test R2: -0.0341


In [13]:
## Save model to be able to reload
model.save("model.h5")
print("Saved model.h5 file successfully")

Saved model.h5 file successfully
